# 1) Load Data
Loading raw game data collected from the Steam API. Data contains nested JSON fields that need to be flattened before analysis.

In [242]:
import pandas as pd
import ast

In [243]:
data = pd.read_csv("../data/raw/games_raw.csv")

print(data.shape)
print(data.isnull().sum())
display(data.dtypes)

(22171, 11)
steam_appid            0
name                   1
is_free                0
developers            28
publishers           101
price_overview      3441
genres                17
categories             3
release_date           0
recommendations    18209
metacritic         21387
dtype: int64


steam_appid        int64
name                 str
is_free             bool
developers           str
publishers           str
price_overview       str
genres               str
categories           str
release_date         str
recommendations      str
metacritic           str
dtype: object

In [244]:
data.head()

,steam_appid,name,is_free,developers,publishers,price_overview,genres,categories,release_date,recommendations,metacritic
0,10,Counter-Strike,False,['Valve'],['Valve'],"{'currency': 'USD', 'initial': 579, 'final': 5...","[{'id': '1', 'description': 'Action'}]","[{'id': 1, 'description': 'Multi-player'}, {'i...","{'coming_soon': False, 'date': '1 Nov, 2000'}",{'total': 168140},"{'score': 88, 'url': 'https://www.metacritic.c..."
1,20,Team Fortress Classic,False,['Valve'],['Valve'],"{'currency': 'USD', 'initial': 499, 'final': 4...","[{'id': '1', 'description': 'Action'}]","[{'id': 1, 'description': 'Multi-player'}, {'i...","{'coming_soon': False, 'date': 'Apr 1, 1999'}",{'total': 6947},NaN
2,30,Day of Defeat,False,['Valve'],['Valve'],"{'currency': 'USD', 'initial': 499, 'final': 4...","[{'id': '1', 'description': 'Action'}]","[{'id': 1, 'description': 'Multi-player'}, {'i...","{'coming_soon': False, 'date': 'May 1, 2003'}",{'total': 4426},"{'score': 79, 'url': 'https://www.metacritic.c..."
3,40,Deathmatch Classic,False,['Valve'],['Valve'],"{'currency': 'USD', 'initial': 499, 'final': 4...","[{'id': '1', 'description': 'Action'}]","[{'id': 1, 'description': 'Multi-player'}, {'i...","{'coming_soon': False, 'date': 'Jun 1, 2001'}",{'total': 2420},NaN
4,50,Half-Life: Opposing Force,False,['Gearbox Software'],['Valve'],"{'currency': 'USD', 'initial': 499, 'final': 4...","[{'id': '1', 'description': 'Action'}]","[{'id': 2, 'description': 'Single-player'}, {'...","{'coming_soon': False, 'date': 'Nov 1, 1999'}",{'total': 24704},NaN


# 2) Pre-processing Steps
Extracting relevant values from nested dictionary columns, converting price to USD, standardizing date format, and dropping columns that are irrelevant to the analysis goal.

## 2.1) Parse Nested Columns
Several columns contain nested JSON stored as strings, thus extracting the relevant values makes them usable for analysis.

In [245]:
data["recommendations"] = data["recommendations"].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)
data["recommendations"] = data["recommendations"].apply(lambda x: x.get("total") if not pd.isna(x) else None)

data["metacritic"] = data["metacritic"].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)
data["metacritic"] = data["metacritic"].apply(lambda x: x.get("score") if not pd.isna(x) else None)

## 2.2) Clean Developers
Converted developers column from a list of names to a count as the number of developers serves as a proxy for studio size, distinguishing indie solo or small team projects from large studio productions. The raw developer names are perserved in the games_raw.csv if needed for future analysis.

In [246]:
data["developers"] = data["developers"].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)
data["developers"] = data["developers"].apply(lambda x: len(x) if x is not None else 0)

## 2.3) Clean Genres
Extracts genre description from nested dictionaries, standardizes inconsistent tags, and filters to a defined set of valid English genres.

In [247]:
data["genres"] = data["genres"].str.replace("Free-to-play", "Free To Play")
data["genres"] = data["genres"].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)
data["genres"] = data["genres"].apply(lambda x: [desc.get("description") for desc in x] if x is not None else None)

In [248]:
# Verification only - confirms non-English genre count before filtering
NON_ENGLISH = ["Aventura", "独立", "竞速", "模拟", "动作", "冒险", "角色扮演", "抢先体验", "Akční", "Strategické", "Aksiyon", "Macera", "RYO", "Azione", "Avventura", "Казуальные игры", 
               "Инди", "Indépendant", "Stratégie", "Aventure", "Abenteuer", "Simuladores", "Actie", "Eventyr", "休闲", "策略"]

display(data["genres"].explode().unique())

count = data["genres"].apply(lambda x: any(i in NON_ENGLISH for i in x) if x is not None else False).sum()
print(f"Games with non-English genres: {count}")

<StringArray>
[               'Action',          'Free To Play',              'Strategy',
             'Adventure',                 'Indie',            'Simulation',
                   'RPG',                'Casual',                'Racing',
 'Massively Multiplayer',                'Nudity',               'Violent',
                'Sports',                     nan,          'Early Access',
                  'Gore',        'Sexual Content',              'Aventura',
            'Accounting', 'Design & Illustration',             'Education',
     'Software Training',             'Utilities',      'Game Development',
                    '独立',                    '竞速',                    '模拟',
                    '动作',                    '冒险',                  '角色扮演',
                  '抢先体验',                 'Akční',           'Strategické',
               'Aksiyon',                'Macera',                   'RYO',
                'Azione',             'Avventura',       'Казуальные игры'

Games with non-English genres: 15


In [249]:
# Excluded non-genre tags: content descriptors (Gore, Violent, Nudity, Sexual Content), software categories (Photo Editing, Design & Illustrations, Video Production Animation & 
# Modeling, Accounting, Software Training, Game Development, Utilities), and monetization models (Free To Play) which are captured separately via price_usd = 0 in the games table.
# Early Access is retained as it represents a legitimate launch strategy for indie studios.
VALID_GENRES = ["Action", "Strategy", "Adventure", "Indie", "Simulation", "RPG", "Casual", "Racing", "Massively Multiplayer", "Sports", "Early Access"]

# 15 games with non-English genre tags are excluded from genre analysis which is less than 0.1% of the dataset.
data["genres"] = data["genres"].apply(lambda x: [valid for valid in x if valid in VALID_GENRES] or None if x is not None else None)

## 2.4) Clean Price

In [250]:
data["price_overview"] = data["price_overview"].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)
data["price_overview"] = (data["price_overview"].apply(lambda x: 0 if pd.isna(x) else x.get("final") if x.get("currency") == "USD" else None) / 100)

## 2.5) Format Dates

In [251]:
data["release_date"] = data["release_date"].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)
data["release_date"] = data["release_date"].apply(lambda x: x.get("date") if not pd.isna(x) else None)
data["release_date"] = pd.to_datetime(data["release_date"], format="mixed", errors="coerce")

## 2.6) Drop and Rename Columns

Rows missing values in price_usd, genres, and release_date were dropped as these fields are central to the goal, which reduced the data from 22,171 to 20,584 rows. Metacritic scores were retained as null rather than dropped as that would reduce the size of the data significantly and impact future analysis. Dropping these would introduce significant survivorship bias toward larger, more established titles. 17,106 games have no recommendation data, which was confirmed against Steam store pages.

In [252]:
data = data.rename(columns={"price_overview": "price_usd", "developers": "developer_count"})

data = data.drop(columns=["is_free", "categories", "publishers"])
data = data.dropna(subset=["price_usd", "genres", "release_date"])

In [253]:
print(data.shape)
print(data.isnull().sum())
display(data.dtypes)

(20584, 8)
steam_appid            0
name                   0
developer_count        0
price_usd              0
genres                 0
release_date           0
recommendations    17106
metacritic         19887
dtype: int64


steam_appid                 int64
name                          str
developer_count             int64
price_usd                 float64
genres                     object
release_date       datetime64[us]
recommendations           float64
metacritic                float64
dtype: object

In [254]:
data.head()

,steam_appid,name,developer_count,price_usd,genres,release_date,recommendations,metacritic
0,10,Counter-Strike,1,5.79,[Action],2000-11-01,168140.0,88.0
1,20,Team Fortress Classic,1,4.99,[Action],1999-04-01,6947.0,NaN
2,30,Day of Defeat,1,4.99,[Action],2003-05-01,4426.0,79.0
3,40,Deathmatch Classic,1,4.99,[Action],2001-06-01,2420.0,NaN
4,50,Half-Life: Opposing Force,1,4.99,[Action],1999-11-01,24704.0,NaN


# 3) Save Data
Saves the clean dataset to data/processed/ for use in SQL analysis and Power BI dashboard.

In [255]:
data.to_csv("../data/processed/games_clean.csv", index=False)